In [1]:
import mlflow
import mlflow.sklearn

from fraudguard.config import load_settings
from fraudguard.data.load import load_dataset
from fraudguard.data.split import temporal_split
from fraudguard.models.metrics import evaluate

settings = load_settings()
mlflow.set_tracking_uri(settings.mlflow.tracking_uri)

# 1. Load the model by alias, not by run or version number
model = mlflow.sklearn.load_model("models:/fraudguard@champion")

# 2. Read the decision threshold stored with that model version
version = mlflow.MlflowClient().get_model_version_by_alias("fraudguard", "champion")
threshold = float(version.tags["threshold"])
print(f"Champion: fraudguard v{version.version}, threshold {threshold:.4f}")

# 3. Evaluate on validation with that threshold
valid = temporal_split(load_dataset(settings), settings.split).valid
proba = model.predict_proba(valid)[:, 1]
print(evaluate(valid["isFraud"], proba, threshold=threshold))

# 4. One raw transaction in, one decision out: what /predict will do
one = valid.iloc[[0]]
p = model.predict_proba(one)[0, 1]
print(f"fraud_probability={p:.4f}, prediction={int(p >= threshold)}, label={one['isFraud'].item()}")

Champion: fraudguard v1, threshold 0.1850
EvaluationResult(pr_auc=0.6086054983466844, roc_auc=0.9251534381475378, threshold=0.185039, precision=0.6505421184320267, recall=0.5266711681296421, f1=0.582089552238806, fnr=0.4733288318703579, g_mean=0.7220187917191034, recall_at_p50=0.6070222822417286)
fraud_probability=0.0087, prediction=0, label=0
